# Test in the trajectory is within safeset or not (Lift task [simulation])

In [ ]:
import os
import sys
import torch
import dill
import numpy as np
import collections
import imageio
import plotly.graph_objects as go

# Import workspace and utilities
from diffusion_policy.workspace.train_diffusion_unet_hybrid_workspace import TrainDiffusionUnetHybridWorkspace
from diffusion_policy.env_runner.robomimic_image_runner import RobomimicImageRunner
from diffusion_policy.gym_util.async_vector_env import AsyncVectorEnv
from diffusion_policy.gym_util.sync_vector_env import SyncVectorEnv
from diffusion_policy.gym_util.multistep_wrapper import MultiStepWrapper
from diffusion_policy.gym_util.video_recording_wrapper import VideoRecordingWrapper, VideoRecorder
from diffusion_policy.model.common.rotation_transformer import RotationTransformer
from diffusion_policy.policy.base_image_policy import BaseImagePolicy
from diffusion_policy.common.pytorch_util import dict_apply
from diffusion_policy.env_runner.base_image_runner import BaseImageRunner
from diffusion_policy.env.robomimic.robomimic_image_wrapper import RobomimicImageWrapper

import robomimic.utils.file_utils as FileUtils
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils

from scipy.spatial.transform import Rotation as R

########################################
# Safe Set Loading and Checking Helpers
########################################

def normalize_quaternion(q):
    norm = np.linalg.norm(q)
    if norm < 1e-6:
        return np.array([0, 0, 0, 1])
    return q / norm

def pose7d_to_6d(pose7d):
    """
    Convert a 7D pose [x, y, z, qx, qy, qz, qw] to a 6D pose:
      - Position: x, y, z
      - Orientation: 3D rotation vector (minimal representation)
    """
    pos = pose7d[:3]
    quat = normalize_quaternion(pose7d[3:7])
    rotvec = R.from_quat(quat).as_rotvec()
    return np.hstack([pos, rotvec])

def load_safe_set(file_path):
    """
    Load the safe set from file. The file is assumed to contain:
       - safe_set: (N,6) safe poses in minimal representation.
       - hull_equations: (m,7) array of half-space equations [A (6 values), b].
       - hull_vertices: indices (for facets) [unused here].
    """
    data = np.load(file_path)
    safe_set = data["safe_set"]
    hull_equations = data["hull_equations"]
    hull_vertices = data["hull_vertices"]
    return safe_set, hull_equations, hull_vertices

# Update the safe set file path here.
safe_set, safe_hull_equations, _ = load_safe_set("/Riad/vivid123/safe_set_6d_simulation.npz")

def is_pose_in_safe_set_6d(query_6d, hull_equations, tol=1e-8):
    """
    Check if a 6D pose is inside the convex hull.
    For each facet defined by A and b, we now check if:
         A.dot(query_6d) + b >= -tol
    This reversal may be necessary depending on the orientation of hull_equations.
    Returns (inside, distances).
    """
    A = hull_equations[:, :-1]  # shape (m,6)
    b = hull_equations[:, -1]   # shape (m,)
    norms = np.linalg.norm(A, axis=1)
    distances = (np.dot(A, query_6d) + b) / norms
    inside = np.all(distances <= tol)
    return inside, distances



########################################
# Delta-to-Absolute Pose Conversion
########################################

from scipy.spatial.transform import Rotation as R
import numpy as np

def apply_delta_pose(current_pose, delta_pose):
    """
    Apply a 7D delta to a current 7D pose.
    
    Parameters:
      current_pose: 7D pose [x, y, z, qx, qy, qz, qw]
      delta_pose: 7D delta [dx, dy, dz, dqx, dqy, dqz, dqw]
      
    Returns:
      new_pose: 7D absolute pose.
    """
    # Update the position by adding translation deltas.
    new_pos = current_pose[:3] + delta_pose[:3]
    
    # Convert current and delta rotations (quaternions) to Rotation objects.
    base_rot = R.from_quat(current_pose[3:])
    delta_rot = R.from_quat(delta_pose[3:])
    
    # Compose rotations (here we assume that the delta is applied in the local frame)
    new_rot = base_rot * delta_rot  # If this order is not correct, try: new_rot = delta_rot * base_rot
    
    new_quat = new_rot.as_quat()
    return np.concatenate([new_pos, new_quat])



########################################
# Environment and Rollout Helpers
########################################

# Frame stacker for temporal context.
class FrameStackForTrans:
    def __init__(self, num_frames):
        self.num_frames = num_frames
        self.obs_history = {}
    
    def reset(self, init_obs):
        self.obs_history = {}
        for k in init_obs:
            self.obs_history[k] = collections.deque([init_obs[k][None] for _ in range(self.num_frames)], maxlen=self.num_frames)
        return {k: np.concatenate(self.obs_history[k], axis=0) for k in self.obs_history}
    
    def add_new_obs(self, new_obs):
        for k in new_obs:
            if 'timesteps' in k or 'actions' in k:
                continue
            self.obs_history[k].append(new_obs[k][None])
        return {k: np.concatenate(self.obs_history[k], axis=0) for k in self.obs_history}

# Environment wrapper to inject a dummy 'robot0_eye_in_hand_image'
class DummyObsWrapper:
    def __init__(self, env):
        self.env = env
        self.required_key = 'robot0_eye_in_hand_image'
        self.image_shape = (3, 84, 84)  # adjust if needed
    
    def reset(self):
        obs = self.env.reset()
        if self.required_key not in obs:
            obs[self.required_key] = np.zeros(self.image_shape, dtype=np.float32)
        return obs
    
    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        if self.required_key not in obs:
            obs[self.required_key] = np.zeros(self.image_shape, dtype=np.float32)
        return obs, reward, done, info
    
    def render(self, *args, **kwargs):
        return self.env.render(*args, **kwargs)
    
    def __getattr__(self, name):
        return getattr(self.env, name)

def create_env(env_meta, shape_meta, enable_render=True):
    modality_mapping = collections.defaultdict(list)
    for key, attr in shape_meta['obs'].items():
        modality_mapping[attr.get('type', 'low_dim')].append(key)
    ObsUtils.initialize_obs_modality_mapping_from_dict(modality_mapping)
    env = EnvUtils.create_env_from_metadata(
        env_meta=env_meta,
        render=False,
        render_offscreen=enable_render,
        use_image_obs=enable_render,
    )
    return env

def undo_transform_action(action, rotation_transformer):
    raw_shape = action.shape
    if raw_shape[-1] == 20:
        action = action.reshape(-1, 2, 10)
    d_rot = action.shape[-1] - 4
    pos = action[..., :3]
    rot = action[..., 3:3+d_rot]
    gripper = action[..., -1:]
    rot = rotation_transformer.inverse(rot)
    uaction = np.concatenate([pos, rot, gripper], axis=-1)
    if raw_shape[-1] == 20:
        uaction = uaction.reshape(*raw_shape[:-1], 14)
    return uaction

########################################
# Rollout Inference Function with Safe Set Check
########################################

def rollout_diffusion(env, policy, rotation_transformer, n_obs_steps, n_action_steps, max_steps, return_imgs=False):
    keys_select = ['robot0_eye_in_hand_image', 'agentview_image', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_gripper_qpos']
    imgs = []
    imgs_eye = []
    framestacker = FrameStackForTrans(n_obs_steps)
    obs = env.reset()
    policy.reset()
    obs = framestacker.reset(obs)
    done = False
    success = False
    step = 0
    trajectory = []  # to store the absolute positions for visualization

    while not done:
        np_obs_dict = {key: obs[key][None, :] for key in keys_select if key in obs}
        obs_dict = dict_apply(np_obs_dict, lambda x: torch.from_numpy(x).to(device))
        with torch.no_grad():
            action_dict = policy.predict_action(obs_dict)
        np_action_dict = dict_apply(action_dict, lambda x: x.detach().cpu().numpy())
        env_action = np_action_dict['action']
        env_action = undo_transform_action(env_action, rotation_transformer)
        env_action = env_action.squeeze()  # assuming shape (N, action_dim)

        for act in env_action:
            # Use only the latest observation from the frame stack.
            current_pose = np.concatenate([obs['robot0_eef_pos'][-1], obs['robot0_eef_quat'][-1]])
    
            # Here we assume act[:7] represents the delta pose:
            # [dx, dy, dz, dqx, dqy, dqz, dqw]
            delta_pose = act[:7]
            
            # Apply the delta to get the predicted absolute pose.
            predicted_pose = apply_delta_pose(current_pose, delta_pose)
            
            # predicted_pose_6d = pose7d_to_6d(predicted_pose)
            # inside, distances = is_pose_in_safe_set_6d(predicted_pose_6d, safe_hull_equations, tol=1e-8)
            # print("Predicted pose safe set check: inside?", inside)
            # print("Predicted Pose 7D:", predicted_pose)

            # noise = np.random.normal(loc=0.0, scale=0.4, size=act.shape)
            # act_noisy = act + noise

            next_obs, reward, done, info = env.step(act)

            # Now print the actual pose from the environment.
            env_pose_7d = np.concatenate([next_obs['robot0_eef_pos'], next_obs['robot0_eef_quat']])
            print("Env Pose 7D after step:", env_pose_7d)
            predicted_pose_6d = pose7d_to_6d(env_pose_7d)
            inside, distances = is_pose_in_safe_set_6d(predicted_pose_6d, safe_hull_equations, tol=1e-1)
            print("Predicted pose safe set check: inside?", inside)
            # print("Predicted Pose 7D:", predicted_pose)

            # # Optionally, also compare the difference
            # diff_7d = env_pose_7d - predicted_pose
            # print("Difference (Env Pose - Predicted Pose):", diff_7d)

            # For visualization, store the current absolute position.
            trajectory.append(current_pose[:3])

            if return_imgs:
                img = env.render(mode="rgb_array", height=512, width=512, camera_name="agentview")
                img_eye = env.render(mode="rgb_array", height=512, width=512, camera_name="robot0_eye_in_hand")
                imgs.append(img)
                imgs_eye.append(img_eye)

            # noise = np.random.normal(loc=0.0, scale=0.4, size=act.shape)
            # act_noisy = act + noise
            next_obs, reward, done, info = env.step(act)
            success = env.is_success()["task"]
            step += 1
            if step == max_steps:
                done = True
                break
            obs = framestacker.add_new_obs(next_obs)
            if done or success:
                done = True
                break
        if done:
            break
    return success, imgs, imgs_eye, np.array(trajectory)

########################################
# Visualization Helper for Rollout Trajectory
########################################

def visualize_trajectory(trajectory, safe_set_file="/Riad/vivid123/safe_set_6d_simulation.npz", tol=1e-8):
    """
    Visualize the rollout trajectory (positions) along with the safe set convex hull.
    """
    # Load safe set.
    data = np.load(safe_set_file)
    safe_set = data["safe_set"]
    safe_positions = safe_set[:, :3]
    # Build 3D convex hull on safe positions.
    from scipy.spatial import ConvexHull
    hull_3d = ConvexHull(safe_positions)
    
    fig = go.Figure()
    
    # Plot safe set hull.
    fig.add_trace(go.Mesh3d(
        x=safe_positions[:, 0],
        y=safe_positions[:, 1],
        z=safe_positions[:, 2],
        i=hull_3d.simplices[:, 0],
        j=hull_3d.simplices[:, 1],
        k=hull_3d.simplices[:, 2],
        opacity=0.3,
        color='lightblue',
        name='Safe Set Hull'
    ))
    
    # Plot trajectory.
    fig.add_trace(go.Scatter3d(
        x=trajectory[:, 0],
        y=trajectory[:, 1],
        z=trajectory[:, 2],
        mode='lines+markers',
        marker=dict(size=4, color='green'),
        line=dict(color='gray', width=2),
        name='Rollout Trajectory'
    ))
    
    fig.update_layout(
        title="Rollout Trajectory on Safe Set",
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z")
    )
    fig.show()

########################################
# Main Execution
########################################

# Set device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Set checkpoint and dataset paths (update these paths as needed).
checkpoint_path = "/Riad/diffusion_policy/data/outputs/Riad_sim_lift_ph_full_2025_03_16_16_54_12/checkpoints/after_train_200_epochs.ckpt"
dataset_path = "/Riad/diffusion_policy/full_image_low_lift_ph.hdf5"  # used for env metadata.

# Load checkpoint payload.
with open(checkpoint_path, 'rb') as f:
    payload = torch.load(f, pickle_module=dill)
cfg = payload['cfg']

# Build workspace and load the payload.
workspace = TrainDiffusionUnetHybridWorkspace(cfg, output_dir=None)
workspace.load_payload(payload, exclude_keys=None, include_keys=None)

# Select policy (use EMA model if enabled).
policy = workspace.model
if cfg.training.use_ema:
    policy = workspace.ema_model
policy.to(device)
policy.eval()
print("Policy loaded and set to eval mode.")

# Get environment metadata.
env_meta = FileUtils.get_env_metadata_from_dataset(dataset_path)
env_meta['env_kwargs']['use_object_obs'] = False  # disable object state observation

# Set absolute action mode if needed and initialize rotation transformer.
abs_action = True
rotation_transformer = None
if abs_action:
    env_meta['env_kwargs']['controller_configs']['control_delta'] = True
    rotation_transformer = RotationTransformer('axis_angle', 'rotation_6d')

# Define shape metadata.
shape_meta = {
    'obs': {
        'robot0_eye_in_hand_image': {'shape': [3, 84, 84], 'type': 'rgb'},
        'agentview_image': {'shape': [3, 84, 84], 'type': 'rgb'},
        'robot0_eef_pos': {'shape': [3]},
        'robot0_eef_quat': {'shape': [4]},
        'robot0_gripper_qpos': {'shape': [2]},
        'object': {'shape': [1]}  # adjust if needed
    },
    'action': {
        'shape': [10]
    }
}

# Create the environment.
raw_env = create_env(env_meta=env_meta, shape_meta=shape_meta, enable_render=True)
print("Created environment with name:", env_meta.get("name", "Unknown"))
if hasattr(raw_env, "action_space"):
    print("Action size is", raw_env.action_space.shape[0])
print("Original env observation keys:", list(raw_env.reset().keys()))

# Wrap the environment to inject dummy 'robot0_eye_in_hand_image' if missing.
env = DummyObsWrapper(raw_env)

# Inference parameters.
n_obs_steps = cfg.dataset_obs_steps if hasattr(cfg, "dataset_obs_steps") else 2
n_action_steps = cfg.n_action_steps if hasattr(cfg, "n_action_steps") else 8
max_steps = 400   # maximum steps per rollout
n_trials = 5      # number of inference trials
fps = 20          # frames per second for the output video

# Run trials and optionally save video for each trial.
trial_success = []
all_traj = []
for i in range(n_trials):
    print(f"Trial {i+1}/{n_trials}...")
    success, imgs, imgs_eye, traj = rollout_diffusion(env, policy, rotation_transformer,
                                                       n_obs_steps, n_action_steps, max_steps,
                                                       return_imgs=True)
    trial_success.append(success)
    all_traj.append(traj)
    print(f"Trial {i+1} success: {success}")
    
    if imgs:
        video_filename = f"trial_{i+1}_output.mp4"
        imageio.mimwrite(video_filename, imgs, fps=fps, quality=8)
        print(f"Saved video: {video_filename}")
    if imgs_eye:
        video_filename_eye = f"trial_{i+1}_output_eye.mp4"
        imageio.mimwrite(video_filename_eye, imgs_eye, fps=fps, quality=8)
        print(f"Saved video: {video_filename_eye}")

mean_success = np.mean(trial_success)
print("Mean success over trials:", mean_success)




# visualize

In [ ]:
# For visualization, we plot the trajectory from the first trial.
if all_traj:
    visualize_trajectory(all_traj[0], safe_set_file="/Riad/vivid123/safe_set_6d_simulation.npz")